# Direct RAW pixel activity explorer

Read _all decoded camera events_ for **any two fixed pixels** in a selected time interval. This notebook reads `trial.raw` directly through Metavision's Python `EventsIterator`; it does not use `pixels.csv` or moving 5×5 patches. The first column shows 5×5 context patches chosen from RAW event activity; the middle column shows every decoded event at each selected pixel; the right column shows same-polarity timing. The spatial patches are for orientation only: they do not limit pixel activity or timing.

Place `trial.raw` beside this notebook, or edit `RAW_PATH`. Run this in a Python environment with the OpenEB/Metavision **Python** bindings, `numpy`, `matplotlib`, and `ipywidgets`. If the time slider stops before your recording ends, increase `MAX_TIME_MS` below. Short windows (20–100 ms) keep decoding and memory use manageable.

**Important:** The notebook replays the C++ per-pixel classifier in Python from RAW events and labels accepted events. The ±1.5% band in the third column is a visual guide; acceptance also depends on polarity history, burst suppression and candidate state. The two selected positions remain fixed while the target may move. The initial controls match the supplied 596 Hz reference screenshot; change them for another recording.


In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output
from matplotlib.patches import Circle
try:
    from metavision_core.event_io import EventsIterator
except ImportError as exc:
    raise ImportError('Run this notebook in your OpenEB environment with metavision_core Python bindings.') from exc

RAW_PATH = Path('trial.raw')
MAX_TIME_MS = 6000       # Change if your recording is longer than 6 seconds.
MAX_CACHED_EVENTS = 8_000_000  # Reduce the selected interval if this limit is reached.
WARMUP_US = 50_000       # Context for intervals beginning at the slider's left edge.
NEIGHBORHOOD_RADIUS = 80 # Limits cached context around chosen pixels, not their classification.
PATCH_RADIUS = 2          # Show 5×5 RAW context patches around active sites.
MAP_RADIUS = 44           # The first column shows a compact view near each selected pixel.
if not RAW_PATH.is_file():
    raise FileNotFoundError(f'Change RAW_PATH to your recording: {RAW_PATH.resolve()}')
print('RAW:', RAW_PATH.resolve())

RAW: /workspace/trial.raw


/usr/local/lib/python3.12/dist-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
def _open_iterator(start_ms, end_ms):
    """Round the requested window to valid EventsIterator start_ts/delta_t boundaries."""
    start_us = int(round(start_ms * 1000))
    end_us = int(round(end_ms * 1000))
    if end_us <= start_us:
        raise ValueError('The selected interval must have positive duration.')
    delta_t = 10_000
    read_start = max(0, start_us - WARMUP_US)
    read_start -= read_start % delta_t  # start_ts must be a multiple of delta_t
    iterator = EventsIterator(input_path=str(RAW_PATH), start_ts=read_start,
        max_duration=end_us-read_start, mode='delta_t', delta_t=delta_t,
        relative_timestamps=False)
    height,width = iterator.get_size()
    if not height or not width:
        raise RuntimeError('The RAW reader did not report sensor dimensions.')
    return iterator,height,width,read_start,start_us,end_us


def read_raw_interval(start_ms, end_ms, targets, radius=NEIGHBORHOOD_RADIUS):
    """Decode one RAW interval plus preceding timing context; retain only events near the given (x,y) targets."""
    iterator,height,width,read_start,start_us,end_us=_open_iterator(start_ms,end_ms)
    chunks=[]
    count=0
    for batch in iterator:
        # SDK event arrays expose t, x, y, p; preserve absolute timestamps.
        if len(batch)==0: continue
        keep=(batch['t']>=read_start)&(batch['t']<end_us)
        # Drop events far from every selected pixel; the sensor-wide count is not needed.
        near=np.zeros(len(batch),dtype=bool)
        bx_,by_=batch['x'].astype(np.int32),batch['y'].astype(np.int32)
        for tx,ty in targets:
            near |= (np.abs(bx_-tx)<=radius)&(np.abs(by_-ty)<=radius)
        keep &= near
        part=batch[keep]
        if len(part):
            count += len(part)
            if count > MAX_CACHED_EVENTS:
                raise MemoryError(f'{count:,} events exceed cache limit; shorten the time interval.')
            chunks.append(part.copy())
    if chunks:
        events=np.concatenate(chunks)
    else:
        events=np.empty(0,dtype=[('t','<i8'),('x','<u2'),('y','<u2'),('p','u1')])
    active=events[(events['t']>=start_us)&(events['t']<end_us)]
    if len(active):
        flat=active['y'].astype(np.int64)*width+active['x'].astype(np.int64)
        counts=np.bincount(flat,minlength=height*width).reshape(height,width)
    else:
        counts=np.zeros((height,width),dtype=np.int64)
    return events,counts,(height,width),(start_us,end_us)


# Match the frequency windows and state transitions in live_frequency_center.cpp.
# Hz: (accepted n*T intervals in microseconds, absolute tolerances in microseconds).
FREQUENCIES = {
    165: ((6061,12121), (61,121)),              # C++: about ±1%
    366: ((2732,5464,8197,10929), (41,82,123,164)),
    596: ((1678,3356,5034,6711), (25,50,76,101)),
}
BURST_US=300
MAX_DT_US=25000
TIMEOUT_TICKS=(20000+255)//256
DISPLAY_TOLERANCE=0.015  # Visual ±1.5% band; classifier uses FREQUENCIES above.

def _matches(dt,hz):
    return any(expected-tolerance <= dt <= expected+tolerance
               for expected,tolerance in zip(*FREQUENCIES[hz]))

def _acquire(dt):
    best=None
    for hz,(expected,tolerances) in FREQUENCIES.items():
        for e,tol in zip(expected,tolerances):
            if e-tol<=dt<=e+tol:
                score=abs(dt-e)/e
                if best is None or score<best[0]:
                    best=(score,hz)
    return 0 if best is None else best[1]

def classify_pixel_events(pixel):
    """Replay the C++ per-pixel candidate/rise/fall logic in timestamp order."""
    last={0:0,1:0}; rings={0:[],1:[]}; candidate=0; support_tick=0
    dt_ms=np.zeros(len(pixel),dtype=float)
    accepted=np.zeros(len(pixel),dtype=np.int16)
    reasons=[]
    for i,(timestamp,polarity) in enumerate(zip(pixel['t'],pixel['p'])):
        now=int(timestamp); p=int(polarity)
        tick=(now>>8)&0xffff
        if candidate and support_tick and ((tick-support_tick)&0xffff)>TIMEOUT_TICKS:
            rings={0:[],1:[]};candidate=0;support_tick=0
        if last[p] and now-last[p]<BURST_US:
            dt_ms[i]=(now-last[p])/1000
            reasons.append('burst');continue
        previous=last[p];last[p]=now
        if previous==0:
            reasons.append('first_polarity_event');continue
        dt=now-previous;dt_ms[i]=dt/1000
        if dt>MAX_DT_US:
            reasons.append('interval_too_long');continue
        if candidate:
            if _matches(dt,candidate):
                rings[p]=(rings[p]+[dt])[-2:]
                support_tick=tick
                if len(rings[0])>=2 and len(rings[1])>=2:
                    accepted[i]=candidate;reasons.append('passed')
                else:reasons.append('candidate_match')
                continue
            rings[p]=[]
            other=_acquire(dt)
            if other and other!=candidate:
                rings={0:[],1:[]};candidate=other;support_tick=tick
                rings[p]=[dt];reasons.append('candidate_switch');continue
            if not rings[0] and not rings[1]:
                rings={0:[],1:[]};candidate=0;support_tick=0
            reasons.append('candidate_mismatch');continue
        other=_acquire(dt)
        if other:
            candidate=other;support_tick=tick;rings={0:[],1:[]}
            rings[p]=[dt];reasons.append('acquired')
        else:reasons.append('no_frequency_match')
    return dt_ms,accepted,np.asarray(reasons,dtype=object)

def pixel_events(events,x,y,bounds):
    """Classify all cached events at a pixel, then display only the selected interval."""
    t0,t1=bounds
    selected=events[(events['x']==x)&(events['y']==y)]
    dt,accepted,reasons=classify_pixel_events(selected)
    visible=(selected['t']>=t0)&(selected['t']<t1)
    return selected[visible],dt[visible],accepted[visible],reasons[visible]

def estimate_led_pixels(start_ms,end_ms,target_hz,radius=NEIGHBORHOOD_RADIUS,num_candidates=6):
    """Suggest target LED using accepted event clusters, then choose the +x outside-r95 site."""
    iterator,height,width,read_start,start_us,end_us=_open_iterator(start_ms,end_ms)
    counts=np.zeros((height,width),dtype=np.int64)
    for batch in iterator:
        if len(batch)==0:continue
        part=batch[(batch['t']>=start_us)&(batch['t']<end_us)]
        if len(part):
            np.add.at(counts,(part['y'].astype(np.int64),part['x'].astype(np.int64)),1)
    if counts.max()==0:raise RuntimeError('No RAW events in this interval.')
    # Bright separated peaks supply spatial proposals; frequency decides which to retain.
    working=counts.copy();candidates=[]
    separation=max(15,min(40,radius//2))
    for _ in range(num_candidates):
        if working.max()==0:break
        py,px=np.unravel_index(np.argmax(working),working.shape)
        candidates.append((int(px),int(py)))
        working[max(0,py-separation):min(height,py+separation+1),
                max(0,px-separation):min(width,px+separation+1)]=0
    # Read full timing context near proposals. Classify each sensor pixel independently.
    events,_,_,_=read_raw_interval(start_ms,end_ms,targets=candidates,radius=radius)
    if not len(events):raise RuntimeError('No events near the candidate peaks.')
    pixels=np.stack((events['x'].astype(np.int32),events['y'].astype(np.int32)),axis=1)
    selected_pixel=np.zeros(len(events),dtype=bool)
    for px,py in candidates:
        selected_pixel |= ((np.abs(pixels[:,0]-px)<=radius)&
                           (np.abs(pixels[:,1]-py)<=radius))
    indices=np.nonzero(selected_pixel)[0]
    # Group by pixel, preserving time order within each stream.
    order=np.lexsort((events['t'][indices],events['y'][indices],events['x'][indices]))
    sorted_idx=indices[order]
    xy=pixels[sorted_idx]
    boundaries=np.r_[0,np.flatnonzero(np.any(xy[1:]!=xy[:-1],axis=1))+1,len(sorted_idx)]
    accepted_indices=[]
    for a,b in zip(boundaries[:-1],boundaries[1:]):
        ids=sorted_idx[a:b]
        _,accepted,_=classify_pixel_events(events[ids])
        good=ids[(accepted==target_hz)&(events['t'][ids]>=start_us)]
        if len(good):accepted_indices.append(good)
    if not accepted_indices:
        raise RuntimeError(f'No accepted {target_hz} Hz events near bright candidates. '
                           'Try a longer interval or choose pixels manually.')
    good=events[np.concatenate(accepted_indices)]
    # Choose the proposal with most target-frequency accepted events nearby.
    scores=[int(((np.abs(good['x'].astype(int)-px)<=radius//2)&
                 (np.abs(good['y'].astype(int)-py)<=radius//2)).sum())
            for px,py in candidates]
    peak_x,peak_y=candidates[int(np.argmax(scores))]
    near=(np.abs(good['x'].astype(int)-peak_x)<=radius//2)&         (np.abs(good['y'].astype(int)-peak_y)<=radius//2)
    group=good[near]
    # The C++ center worker uses a 10 ms window of subsampled accepted events.
    # Here all accepted events in the trailing 10 ms are used; no wall-clock scheduling.
    latest=int(group['t'].max())
    group=group[group['t']>=latest-10000]
    if len(group)<4:raise RuntimeError('Too few accepted events in the last 10 ms.')
    cx=float(group['x'].mean());cy=float(group['y'].mean())
    distances=np.hypot(group['x'].astype(float)-cx,group['y'].astype(float)-cy)
    r95=float(np.partition(distances,int(.95*(len(distances)-1)))[int(.95*(len(distances)-1))])
    # The geometric centroid need not itself have any events. Select a nearby
    # measured pixel with accepted events for the center time-series plot.
    all_group=good[near]
    center_distance=np.hypot(all_group['x'].astype(float)-cx,
                             all_group['y'].astype(float)-cy)
    nearby=all_group[center_distance<=max(5.,min(12.,r95/3))]
    if not len(nearby):
        nearby=all_group[center_distance==center_distance.min()]
    unique_xy,accepted_counts=np.unique(
        np.stack((nearby['x'],nearby['y']),axis=1),axis=0,return_counts=True)
    near_distance=np.hypot(unique_xy[:,0]-cx,unique_xy[:,1]-cy)
    best=np.lexsort((near_distance,-accepted_counts))[0]
    pixel_center=(int(unique_xy[best,0]),int(unique_xy[best,1]))
    # Use the actual centroid, not the chosen active pixel, to define r95.
    icx,icy=int(round(cx)),int(round(cy))
    # Pick an active RAW pixel outside r95, close enough to compare against the center.
    # Require three pixels of clearance so the chosen 5×5 context patch is outside.
    outer=max(40.,r95+35.)
    x0=max(2,int(np.floor(cx-outer)));x1=min(width-2,int(np.ceil(cx+outer))+1)
    y0=max(2,int(np.floor(cy-outer)));y1=min(height-2,int(np.ceil(cy+outer))+1)
    yy,xx=np.mgrid[y0:y1,x0:x1]
    distance=np.hypot(xx-cx,yy-cy)
    eligible=(distance>r95+PATCH_RADIUS+1)&(distance<=outer)&(counts[y0:y1,x0:x1]>=4)
    eligible &= (np.abs(xx-icx)>PATCH_RADIUS)|(np.abs(yy-icy)>PATCH_RADIUS)
    if not np.any(eligible):
        raise RuntimeError('No active RAW pixel outside r95 nearby; choose an outside pixel manually.')
    # Choose on the +x side when it contains activity, as in the reference view.
    right=eligible&(xx>cx)
    if np.any(right): eligible=right
    scores=np.where(eligible,counts[y0:y1,x0:x1],-1)
    oy_local,ox_local=np.unravel_index(np.argmax(scores),scores.shape)
    ox,oy=int(x0+ox_local),int(y0+oy_local)
    return pixel_center,(ox,oy),(cx,cy),r95,len(group),latest


In [3]:
# A single image widget holds the current plot. Replacing its bytes cannot
# accumulate Matplotlib output across callbacks or notebook frontends.
from io import BytesIO
from html import escape

if '_raw_pixel_explorer_controls' in globals():
    for old in _raw_pixel_explorer_controls:
        old.unobserve_all()
        old.close()
    _raw_pixel_explorer_ui.close()

frequency=widgets.Dropdown(options=[165,366,596],value=596,description='Target Hz')
interval=widgets.FloatRangeSlider(value=(352.0,673.0),min=0.0,max=float(MAX_TIME_MS),
    step=1.0,description='Time (ms)',readout_format='.0f',continuous_update=False,
    layout=widgets.Layout(width='700px'))
ax=widgets.BoundedIntText(value=226,min=0,max=1279,description='Center x')
ay=widgets.BoundedIntText(value=301,min=0,max=719,description='Center y')
bx=widgets.BoundedIntText(value=255,min=0,max=1279,description='Outside x')
by=widgets.BoundedIntText(value=299,min=0,max=719,description='Outside y')
load_button=widgets.Button(description='Load selected interval',button_style='info')
locate_button=widgets.Button(description='Suggest pixels (approx.)',button_style='warning')
status=widgets.HTML()
figure_widget=widgets.Image(format='png',layout=widgets.Layout(width='100%'))
cache={'key':None,'data':None}

def show_status(message):
    status.value='<pre style="white-space:pre-wrap">'+escape(message)+'</pre>'

def locate_led(_=None):
    show_status('Finding accepted frequency events, center and r95 (may take a moment)…')
    try:
        (px,py),(ox,oy),center,r95,n_samples,latest=estimate_led_pixels(
            *interval.value, target_hz=frequency.value)
    except Exception as exc:
        show_status(type(exc).__name__+': '+str(exc))
        return
    cache['led']={'center':center,'radius':r95,'samples':n_samples,
                  'latest':latest,'frequency':frequency.value,
                  'window':tuple(interval.value)}
    ax.value,ay.value=px,py
    bx.value,by.value=ox,oy
    load_interval(None)

def draw(_=None):
    if cache['data'] is None:
        show_status('Press Load selected interval to decode RAW events.')
        return
    led=cache.get('led')
    if led and (led['frequency']!=frequency.value or led['window']!=tuple(interval.value)):
        led=None
    events,counts,(height,width),(left_us,right_us)=cache['data']
    if cache.get('targets')!=((ax.value,ay.value),(bx.value,by.value)):
        show_status('Pixel positions changed. Press Load selected interval to read these RAW pixels.')
        return
    summaries=[]
    fig,axes=plt.subplots(2,3,figsize=(17,8),sharex='col',sharey='col')
    # Only the contextual map is sampled into 5×5 patches. Every RAW event
    # at the two selected pixel positions still appears in the time plots.
    spatial_ranges=[]
    for row,(name,xw,yw) in enumerate((('Center',ax,ay),('Outside',bx,by))):
        x,y=xw.value,yw.value
        spatial,activity,timing=axes[row]
        if x>=width or y>=height:
            summaries.append(f'{name} pixel ({x},{y}) is outside the {width}x{height} sensor.')
            continue
        radius=MAP_RADIUS
        xmin,xmax=max(0,x-radius),min(width,x+radius+1)
        ymin,ymax=max(0,y-radius),min(height,y+radius+1)
        local=counts[ymin:ymax,xmin:xmax]
        # Show separated local activity peaks and always include the selected
        # pixel patch. These patches represent actual RAW event counts.
        peaks=[(x,y)]
        working=local.copy()
        if led is not None:
            cyy,cxx=np.mgrid[ymin:ymax,xmin:xmax]
            cx,cy=led['center'];r=led['radius']
            distances=np.hypot(cxx-cx,cyy-cy)
            working[(distances>r+PATCH_RADIUS) if row==0
                    else (distances<=r+PATCH_RADIUS)]=0
        for _ in range(42):
            if working.max()<2:break
            py,px=np.unravel_index(np.argmax(working),working.shape)
            px+=xmin;py+=ymin
            peaks.append((int(px),int(py)))
            lx,ly=px-xmin,py-ymin
            working[max(0,ly-4):min(working.shape[0],ly+5),
                    max(0,lx-4):min(working.shape[1],lx+5)]=0
        patch_mask=np.zeros(local.shape,dtype=bool)
        for px,py in peaks:
            lx,ly=px-xmin,py-ymin
            patch_mask[max(0,ly-PATCH_RADIUS):min(local.shape[0],ly+PATCH_RADIUS+1),
                       max(0,lx-PATCH_RADIUS):min(local.shape[1],lx+PATCH_RADIUS+1)]=True
        yy,xx=np.nonzero(patch_mask&(local>0))
        xx=xx+xmin;yy=yy+ymin
        if len(xx):
            spatial.scatter(xx,yy,c=local[yy-ymin,xx-xmin],s=17,
                            cmap='viridis',rasterized=True)
        spatial.scatter([x],[y],marker='x',s=90,c='red',label='Selected pixel',zorder=5)
        if led is not None:
            center=led['center'];r=led['radius']
            spatial.add_patch(Circle(center,r,fill=False,color='orange',lw=1.5,
                                     label='Median r95 (10 ms)'))
            spatial.scatter([center[0]],[center[1]],marker='+',s=70,c='orange')
            spatial_ranges.append((int(center[0]-r),int(center[0]+r),
                                   int(center[1]-r),int(center[1]+r)))
        spatial_ranges.append((xmin,xmax-1,ymin,ymax-1))
        spatial.set(title=f'{name} sampled pixels',ylabel='Sensor y (px)')
        spatial.set_aspect('equal',adjustable='box')
        spatial.legend(fontsize=7,loc='upper right')
        selected,dt,accepted,reasons=pixel_events(events,x,y,(left_us,right_us))
        times=(selected['t'].astype(np.int64)-left_us)/1000
        good=accepted==frequency.value
        for polarity,color in ((0,'#0072b2'),(1,'#d55e00')):
            mask=selected['p']==polarity
            rejected=mask&~good
            activity.scatter(times[rejected],np.full(rejected.sum(),polarity),
                s=19,c='#0072b2',alpha=.8,
                label='Raw, rejected' if polarity==0 else None)
            valid=mask&np.isfinite(dt)
            timing.scatter(times[valid],dt[valid],s=19,c=color,alpha=.8,
                label='OFF' if polarity==0 else 'ON')
        activity.scatter(times[good],selected['p'][good],s=65,marker='x',
                         c='#009e73',label='Accepted')
        activity.set(yticks=[0,1],yticklabels=['OFF','ON'],ylim=(-.3,1.3),
                     title=f'{name} pixel events')
        period_ms=1000/frequency.value
        timing.axhspan(period_ms*(1-DISPLAY_TOLERANCE),
            period_ms*(1+DISPLAY_TOLERANCE),color='green',alpha=.15,
            label='±1.5% visual band')
        timing.axhline(period_ms,linestyle='--',color='green',
                       label=f'{frequency.value} Hz period')
        timing.set(title=f'{name} timing',ylabel='Same-polarity Δt (ms)')
        for panel in (activity,timing):
            panel.set_xlim(0,(right_us-left_us)/1000)
            panel.grid(alpha=.2)
            panel.legend(fontsize=8,loc='upper right')
        if len(selected):
            counts_by_reason=dict(zip(*np.unique(reasons,return_counts=True)))
            summaries.append(f'{name} pixel ({x},{y}): {len(selected)} RAW events, '
                  f'{int(good.sum())} accepted as {frequency.value} Hz; '
                  f'reasons: {counts_by_reason}')
        else:
            summaries.append(f'{name} pixel ({x},{y}): no RAW events in this interval.')
    if spatial_ranges:
        left=min(r[0] for r in spatial_ranges)-5
        right=max(r[1] for r in spatial_ranges)+5
        top=min(r[2] for r in spatial_ranges)-5
        bottom=max(r[3] for r in spatial_ranges)+5
        for spatial in axes[:,0]:
            spatial.set_xlim(max(0,left),min(width,right))
            spatial.set_ylim(min(height,bottom),max(0,top))
    axes[-1,0].set_xlabel('Sensor x (px)')
    for panel in axes[-1,1:]:panel.set_xlabel('Time within interval (ms)')
    fig.tight_layout()
    image_bytes=BytesIO()
    try:
        fig.savefig(image_bytes,format='png',dpi=110)
    finally:
        plt.close(fig)
    figure_widget.value=image_bytes.getvalue()  # Replaces one image; no plt.show().
    show_status('\n'.join(summaries))

def load_interval(_):
    show_status('Decoding selected RAW interval…')
    try:
        targets=[(ax.value,ay.value),(bx.value,by.value)]
        data=read_raw_interval(*interval.value, targets=targets)
        cache['data']=data
        cache['targets']=tuple(targets)
        cache['key']=tuple(interval.value)
    except Exception as exc:
        cache['data']=None
        cache.pop('targets',None)
        show_status(type(exc).__name__+': '+str(exc))
        return
    draw()

def interval_changed(_):
    cache['data']=None
    cache.pop('led',None)
    figure_widget.value=b''
    show_status('Time range changed. Press Load selected interval.')

locate_button.on_click(locate_led)
load_button.on_click(load_interval)
frequency.observe(draw,names='value')
interval.observe(interval_changed,names='value')
_raw_pixel_explorer_controls=(frequency,interval,ax,ay,bx,by,load_button,locate_button)
_raw_pixel_explorer_ui=widgets.VBox([
    widgets.HBox([frequency,load_button,locate_button]),interval,
    widgets.HBox([ax,ay]),widgets.HBox([bx,by]),status,figure_widget])
clear_output(wait=True)
display(_raw_pixel_explorer_ui)
show_status('Press Load selected interval to decode RAW events.')


## Interpretation and limits

The upper and lower rows are two fixed pixels over the same interval. The first column shows 5×5 context patches around locally active RAW pixels near each selected point, colored by event count. When r95 is available, center and outside patches are grouped according to their distance from that radius. The spatial map is only a display sample; the other columns include every decoded RAW event at each selected pixel. Blue/orange show OFF/ON raw events and same-polarity intervals; green crosses mark events that passed the notebook's replay of the C++ per-pixel state machine. The shaded green interval is ±1.5% around one selected period. Accepted events can also match integer multiples of the period if cycles were missed.

**Suggest pixels** finds bright proposals, scores accepted events for the selected frequency, computes a centroid and r95 from a trailing 10 ms accepted-event window, chooses a nearby pixel that actually produced accepted events as the selected center, and selects an active RAW pixel to the right of the center beyond r95 and its 5×5 patch clearance (or another side when no pixel is active to the right). The C++ live center worker subsamples every fourth accepted event and publishes at wall-clock intervals, which this offline notebook does not reproduce exactly. A second pixel outside r95 is not necessarily a reflection. Automatic frequency selection can be confused by same-frequency reflections or motion; inspect the spatial plot before drawing conclusions. The first roughly 20 ms after the analysis window starts may differ from a full-file classifier replay if earlier history matters.
